# Module 5: Critic-Refiner: Deploy to Amazon Bedrock AgentCore

**Pattern 3: Critic-Refiner**: Writer drafts the memo, Critic checks it against 5 criteria, revises until APPROVED.

![Critic-Refiner Runtime architecture](./architecture.png)

**What this notebook covers:**
1. Install the AgentCore CLI
2. Configure and deploy
3. Review `main.py`
4. Find the Runtime ARN
5. Invoke the deployed agent with boto3
6. Observability in CloudWatch
7. Cleanup

---

## Step 1: Install the AgentCore CLI

In [ ]:
!uv pip install --system bedrock-agentcore-starter-toolkit bedrock-agentcore strands-agents boto3

In [ ]:
import boto3, json, os

# Set up execution roles for AgentCore runtimes.
# Creates them if missing; uses them if already exist (idempotent).

iam = boto3.client("iam")
sts = boto3.client("sts")
account = sts.get_caller_identity()["Account"]
region  = os.environ.get("AWS_REGION", "us-east-1")
bucket  = f"bedrock-agentcore-deploy-{account}-{region}"

trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow",
                   "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                   "Action": "sts:AssumeRole"}]
})

for role_name, env_var in [
    ("workshop-agentcore-m5-runtime-role",      "AGENTCORE_RUNTIME_ROLE_ARN"),
    ("workshop-agentcore-m5-orchestrator-role", "AGENTCORE_ORCHESTRATOR_ROLE_ARN"),
]:
    try:
        arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
        print(f"  {role_name}")
        print(f"  {arn}\n")
    except iam.exceptions.NoSuchEntityException:
        try:
            arn = iam.create_role(RoleName=role_name,
                                  AssumeRolePolicyDocument=trust,
                                  Description="AgentCore runtime role")["Role"]["Arn"]
            print(f"  Created: {role_name}")
            print(f"  {arn}\n")
        except Exception as e:
            print(f"  Could not create {role_name}: {e}")
            print(f"  Set {env_var} env var to a pre-existing role ARN\n")
            arn = None
    except Exception as e:
        print(f"  {role_name}: {e}\n")
        arn = None

    if arn:
        os.environ[env_var] = arn

---

## Step 2: Deploy

Runs all three runtimes (~3-5 min). The cell below executes `deploy.py` directly so it picks up the role ARNs set in the previous cell.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "deploy.py", "--name-prefix", "m5"],
    capture_output=False,
)
if result.returncode != 0:
    raise SystemExit(f"deploy.py failed with exit code {result.returncode}")

---

## Step 3: Review `main.py`

This is the code that runs inside the Runtime container.

In [ ]:
print(open("main.py").read())

---

## Step 4: Find the Runtime ARN

Run this cell to list all deployed Runtimes and find the ARN for this module.

In [ ]:
import os
import boto3

client_control = boto3.client("bedrock-agentcore-control", region_name=os.environ.get("AWS_REGION", "us-east-1"))
response = client_control.list_agent_runtimes()

print(f"{'Name':<40} {'ARN'}")
print("-" * 100)
for rt in response.get("agentRuntimes", []):
    print(f"{rt['agentRuntimeName']:<40} {rt['agentRuntimeArn']}")

---

## Step 5: Invoke the deployed agent

Paste the ARN from Step 4 into `RUNTIME_ARN` below, then run the cell.

In [ ]:
import os
import boto3, json, uuid
from botocore.config import Config

RUNTIME_ARN = ""  # paste ARN from Step 4 here

if not RUNTIME_ARN:
    raise ValueError("Set RUNTIME_ARN above before running this cell.")

client = boto3.client(
    "bedrock-agentcore",
    region_name=os.environ.get("AWS_REGION", "us-east-1"),
    config=Config(read_timeout=300),   # pipelines can take 60-180s
)

response = client.invoke_agent_runtime(
    agentRuntimeArn=RUNTIME_ARN,
    runtimeSessionId=str(uuid.uuid4()),
    payload=json.dumps({
        "prompt": "NovaCart Premium Tier: Options A ($19.99/mo invite-only), B ($14.99/mo 5% pilot), C ($12.99/mo full launch). Target: +15% CLV in 6 months."
    }).encode(),
    qualifier="DEFAULT",
)

result = json.loads(response["response"].read())
print(result.get("response", result))

---

## Step 6: Observability

After invoking, traces appear in **CloudWatch > X-Ray > Traces** or **Amazon Bedrock > AgentCore > Observability**.

What you see per invocation:
- Root span per `invoke_agent_runtime` call
- Child span per `Agent()` call inside the pipeline
- Tool call spans nested under each agent
- Duration breakdown per stage

No extra configuration needed: `aws-opentelemetry-distro` is in `requirements.txt` and AgentCore installs it automatically.

---

## Step 7: Cleanup

Delete all AWS resources created by this module.

```bash
python cleanup.py --name-prefix m5
```

Verify cleanup:

```bash
aws bedrock-agentcore-control list-agent-runtimes --region us-east-1
```